Prediction of Alzheimer's Diagnosis from Perfusion MRI
HIDS-7006, AI for Health Applications, Spring 2025
Stephanie Araki (solo project)

Classifies cognitively normal (CN), mild cognitive impairment (MCI), and Alzheimer's
disease (AD) using cerebral blood flow (CBF) features from Arterial Spin Labeling (ASL)
MRI, combined with demographic, genetic (APOE4), and cognitive assessment data from ADNI.

Pipeline (as documented in the final project report):
  1. Merge ASL perfusion features (130 FreeSurfer ROIs) with demographic/clinical/cognitive data
  2. Group-aware train/test split (patients, not visits, are held out) to prevent data leakage
  3. Impute, scale, and one-hot encode via a ColumnTransformer pipeline
  4. Compare 6 classifiers via 5-fold group-aware cross-validation
  5. Address class imbalance with SMOTE, tune the winning model, interpret with SHAP

In [ ]:
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
RANDOM_STATE = 42

In [ ]:
CLINICAL_FEATURES = [
    "AGE", "PTGENDER", "PTEDUCAT", "APOE4",
    "MMSE", "CDRSB", "MOCA", "ADAS11", "ADAS13", "LDELTOTAL", "EcogPtTotal", "EcogSPTotal",
]

In [ ]:
CBF_FEATURE_PATTERN = ["MIN", "MAX", "MD", "AVG", "SD"]

### `build_dataset`

In [ ]:
def build_dataset(demo_path: str, asl_path: str) -> pd.DataFrame:
    """Merge ADNI baseline demographic/clinical data with ASL CBF features on RID + VISCODE."""
    demo = pd.read_csv(demo_path)
    asl = pd.read_csv(asl_path)
    merged = demo.merge(asl, on=["RID", "VISCODE"], how="inner")
    return merged

### `build_preprocessing_pipeline`

In [ ]:
def build_preprocessing_pipeline(cbf_columns: list[str]) -> ColumnTransformer:
    """Impute + scale numeric features, one-hot encode categorical features."""
    numeric_features = CLINICAL_FEATURES[:-2] + cbf_columns  # exclude gender (categorical)
    categorical_features = ["PTGENDER"]

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

### `compare_classifiers`

In [ ]:
def compare_classifiers(X, y, groups):
    """5-fold group-aware CV so repeat visits from the same patient never split
    across train/test folds."""
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced"),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(eval_metric="mlogloss", random_state=RANDOM_STATE),
    }
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    results = {}
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv, groups=groups, scoring="accuracy")
        results[name] = (scores.mean(), scores.std())
    return results

### `train_best_model_with_smote_and_tuning`

In [ ]:
def train_best_model_with_smote_and_tuning(X, y):
    """Random Forest was the best-performing model; oversample minority classes
    (CN, AD) with SMOTE and tune via grid search, matching the 86% test accuracy
    reported in the final write-up."""
    pipeline = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
    ])
    param_grid = {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [10, 20, None],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2],
    }
    search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy")
    search.fit(X, y)
    return search.best_estimator_, search.best_params_

### `explain_with_shap`

In [ ]:
def explain_with_shap(model, X_test, feature_names):
    """SHAP feature importance: identifies which cognitive scores and CBF regions
    carry the most predictive signal for each diagnostic class."""
    explainer = shap.TreeExplainer(model.named_steps["model"])
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
    return shap_values

### Run

In [ ]:
if __name__ == "__main__":
    df = build_dataset("data/adni_baseline.csv", "data/adni_asl.csv")

    cbf_columns = [c for c in df.columns if any(c.endswith(f"_{suffix}") for suffix in CBF_FEATURE_PATTERN)]
    preprocessor = build_preprocessing_pipeline(cbf_columns)

    le = LabelEncoder()
    y = le.fit_transform(df["DX_bl"])  # CN=0, MCI=1, AD=2 (or similar, per LabelEncoder ordering)
    groups = df["RID"]  # group by patient so visits from the same subject stay together

    X = preprocessor.fit_transform(df)

    print("5-fold group-aware CV accuracy by model:")
    for name, (mean, std) in compare_classifiers(X, y, groups).items():
        print(f"  {name}: {mean:.3f} +/- {std:.3f}")

    best_model, best_params = train_best_model_with_smote_and_tuning(X, y)
    print(f"\nBest Random Forest params: {best_params}")